# SeeSaw — Notebook 1: Data Preparation

Converts TinyStories + SeeSaw StoryBeatRecord exports into the Gemma 3 instruction format.

**Output:** `gs://seesaw-models/training-data/seesaw_beats_train.jsonl`

**Runtime:** Free Colab T4, ~30 minutes

See `docs/FINE_TUNING.md` for full details.

In [ ]:
# Step 1: Install dependencies
!pip install -q datasets google-cloud-storage huggingface_hub

In [ ]:
# Step 2: Authenticate with GCP (runs browser flow in Colab)
from google.colab import auth
auth.authenticate_user()

import subprocess
subprocess.run(['gcloud', 'config', 'set', 'project', 'seesaw-3e396'], check=True)
print('GCP auth complete')

In [ ]:
# Step 3: Load TinyStories
from datasets import load_dataset

ds = load_dataset('roneneldan/TinyStories', split='train')
sample = ds.shuffle(seed=42).select(range(12000))  # oversample, filter below
print(f'Loaded {len(sample)} stories')

In [ ]:
# Step 4: Safety filter
BANNED_TERMS = [
    'kill', 'die', 'dead', 'blood', 'gun', 'knife', 'weapon',
    'scary', 'nightmare', 'horror', 'alcohol', 'drug', 'sex',
]

def is_safe(text: str) -> bool:
    text_lower = text.lower()
    return not any(term in text_lower for term in BANNED_TERMS)

safe = sample.filter(lambda x: is_safe(x['text']))
print(f'After safety filter: {len(safe)} stories')

In [ ]:
# Step 5: Convert TinyStories to SeeSaw instruction format (Gemma 3 chat template)
import json

SYSTEM_PROMPT = """You are SeeSaw, a gentle storytelling companion for children aged 4-8.
Generate story beats as JSON: {\"story_text\": \"...\", \"question\": \"...\", \"is_ending\": false}"""

def to_instruction_format(story_text: str) -> str:
    sentences = [s.strip() for s in story_text.split('.') if s.strip()]
    if len(sentences) < 2:
        return None
    story = '. '.join(sentences[:3]) + '.'
    question = 'What do you think happens next?'
    payload = json.dumps({'story_text': story, 'question': question, 'is_ending': False})
    return (
        f'<bos><start_of_turn>user\n{SYSTEM_PROMPT}\n\nContinue the story.<end_of_turn>\n'
        f'<start_of_turn>model\n{payload}<end_of_turn>'
    )

tinystories_examples = []
for row in safe:
    formatted = to_instruction_format(row['text'])
    if formatted:
        tinystories_examples.append({'text': formatted})

print(f'TinyStories examples: {len(tinystories_examples)}')

In [ ]:
# Step 6: Load SeeSaw iOS StoryBeatRecord exports (domain-specific gold examples)
# Upload seesaw_training_all.jsonl to Colab via Files panel, or paste the path below.
# These are already in the correct Gemma instruction format — use as-is.

import json, os

IOS_BEATS_FILE = 'seesaw_training_all.jsonl'   # upload to Colab root

ios_examples = []
if os.path.exists(IOS_BEATS_FILE):
    with open(IOS_BEATS_FILE) as f:
        ios_examples = [json.loads(line) for line in f if line.strip()]
    print(f'iOS beat examples: {len(ios_examples)}')
else:
    print(f'⚠ {IOS_BEATS_FILE} not found — continuing with TinyStories only')
    print('  Upload the file via Colab Files panel and re-run this cell.')

In [ ]:
# Step 7: Merge datasets — iOS beats first (higher weight via duplication), then TinyStories
import random

# Duplicate iOS beats 5× to give them more weight in training
weighted_ios = ios_examples * 5

# Fill remaining budget from TinyStories up to 8,000 total
budget = max(0, 8000 - len(weighted_ios))
random.seed(42)
random.shuffle(tinystories_examples)
examples = weighted_ios + tinystories_examples[:budget]
random.shuffle(examples)

print(f'Final dataset: {len(examples)} examples')
print(f'  iOS beats (5×): {len(weighted_ios)}')
print(f'  TinyStories: {min(budget, len(tinystories_examples))}')

In [ ]:
# Step 8: Save to GCS
import json
from google.cloud import storage

GCS_BUCKET = 'seesaw-models'
BLOB_NAME  = 'training-data/seesaw_beats_train.jsonl'

jsonl = '\n'.join(json.dumps(ex) for ex in examples)

client = storage.Client(project='seesaw-3e396')
bucket = client.bucket(GCS_BUCKET)
blob   = bucket.blob(BLOB_NAME)
blob.upload_from_string(jsonl, content_type='application/jsonl')

print(f'Uploaded {len(examples)} examples to gs://{GCS_BUCKET}/{BLOB_NAME}')